# Worm profile: what the calibration measured, and what the feed-forward does with it

Each motor's gear train has a periodic error after the motor (a 6.0° worm) that the MCU can't see. The **worm
profile test** (Speed Calibration page, the **M1-M2-M3-WORM-PROFILE** row) measures all three motors' worms at once
from plate solves, and the **worm feed-forward** corrects for them while tracking. This notebook reads the profile
file (read-only) and answers:

1. **What is in the file?** The tests kept, the profile in use, and what Approve would apply.
2. **Did a test measure the worms?** One test's fit, sample by sample, and each motor's worm on its own.
3. **Do tests agree?** The main question: a profile is worth applying only if the worms repeat wherever and whenever
   they are measured.
4. **Does the pose matter?** M1 and M3 are told apart by the roll.
5. **Is the model right?** 2nd harmonic, backlash, what the fit leaves.
6. **What would Approve change?** The pooled profile against the one in use.
7. **How long does a test take?** Settle times, discarded syncs.
8. **Does a night of tracking agree?** The worm terms in a session's guide corrections, against the calibration.
9. **Twin only:** do the fits recover the simulated worms, and are their error bars honest?

## The test

The tracked target is stepped so that each motor turns its own schedule (33 positions; M1 0.6° steps reversing at
positions 8 and 24, M2 0.45° reversing at 12, M3 0.375° reversing at 16), while sidereal tracking holds the sky. The
different steps advance each motor's worm phase at its own rate (36°, 27°, 22.5° a step), so their worms separate; the
reversals separate their backlash. At each position the client's plate solve and sync is recorded, not applied: the
2-D pointing error (solved minus predicted, horizontal and up, arcsec) and each motor's effect on the pointing there
(J, 3 × 2). If |Roll| is under 20° the test first rotates to ±25° and back afterwards: at Roll 0, M1 and M3 move the
field the same way. Ideal pose: Alt 30–45°, any azimuth.

## The model

On each motor's MCU angle ζ (517, deg, the same every session) with worm phase φ = 360° × ζ / 6.0:

`e(ζ) = a₁ sin φ + b₁ cos φ + a₂ sin 2φ + b₂ cos 2φ` (arcsec of motor angle; true angle = MCU angle + e)

Each sample's 2-D error = Σₘ Jₘ × (eₘ + backlashₘ when moving forward), plus per test a 2-D offset, a quadratic drift in
time and a linear trend across the pointing (`control_worm.profile_design`). The 1st harmonic must reach 4 × its
standard error (SE) for COMPLETED; the 2nd is applied only at 3 × SE. Approve applies the fit pooled over the last 5
completed tests. `A = hypot(a, b)`, phase `p = atan2(b, a)`.

**Phasors.** Sections 3, 6 and 8 plot results as the point (a₁, b₁). The distance between two points is the worm
left uncorrected if one were used to correct the other, so the scatter of the points is the correction to expect.

**Error bars.** On the digital twin the SE has been up to 1.5–2 × optimistic, the pooled fit's most (the samples
are not independent: drift and the pointing model aren't exactly the fitted forms). Section 3 checks this on real
tests: the spread between tests against their SE.

In [1]:
import os, sys, json
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display
sys.path.insert(0, os.path.abspath(os.path.join('.', '..', 'driver')))
import importlib, worm_analysis, analyse_helpers
importlib.reload(worm_analysis); importlib.reload(analyse_helpers)
import control_worm
from control_worm import (WormFeedForward, fit_worm_profile, pooled_profile, MOTORS, POOL_TESTS, MIN_SIGNIFICANCE,
                          MIN_H2_SIGNIFICANCE, MIN_SEPARATION_DEG, SETTLE_ARCSEC, SETTLE_HOLD_S)
from worm_analysis import profile_tests, test_fit_details, wff_coupling, worm_terms, motor_angles, WORM_THETA

# the profile file: the driver's own, or WORM_PROFILE (e.g. one written by the digital twin, with its true worms)
PROFILE = os.environ.get('WORM_PROFILE') or os.path.abspath(os.path.join('..', 'data', 'worm_profile.json'))
TEST = -1                       # the test section 2 looks at (index into the list below; -1 = the latest)
TEMPLATE = 'plotly_dark'
COLOURS = {'M1': 'royalblue', 'M2': 'mediumseagreen', 'M3': 'orange'}

with open(PROFILE) as f:
    raw = json.load(f)
TRUTH = raw.get('twin_truth')                      # only in files written by the twin
ff = WormFeedForward.load(PROFILE)
tests = profile_tests(PROFILE)
print(f"{PROFILE}\n{len(tests)} worm profile tests" + ("   (TWIN FILE: simulated worms)" if TRUTH else ''))
if not tests:
    print("No M1-M2-M3-WORM-PROFILE tests in this file yet: run one from the Speed Calibration page (M1 row).")

def phasor(A, p):
    return A * np.cos(np.radians(p)), A * np.sin(np.radians(p))

def in_use(m):
    # the profile in use: (a1, b1, a2, b2) for motor m (0 where a harmonic isn't in it)
    out = np.zeros(4)
    for j, h in enumerate(ff.harmonics):
        if h in (1, 2):
            out[2 * (h - 1):2 * h] = ff.coef[m][2 * j:2 * j + 2]
    return out

/home/jdm/projects/alpaca-benro-polaris/data/worm_profile.json
0 worm profile tests
No M1-M2-M3-WORM-PROFILE tests in this file yet: run one from the Speed Calibration page (M1 row).


## 1. What is in the file?

Every test kept (the last 15), each fitted on its own: status, positions kept, fit rms, the M1/M3 separation range
(below 10° the M1 and M3 worms are uncertain), each motor's 1st harmonic and its significance (amplitude / SE). Then
the profile in use (what the feed-forward applies now).

In [2]:
rows = []
for i, e in enumerate(tests):
    r, s = e['result'], e['samples']
    row = {'#': i, 'created': e['created'], 'status': e['status'], 'positions': r['checks'].get('positions'),
           'rms "': r['checks'].get('rms_arcsec'), 'sep min°': (r['checks'].get('separation_deg') or [np.nan])[0],
           'az': np.mean([x['a_az'] for x in s]), 'alt': np.mean([x['a_alt'] for x in s])}
    for M, v in r.get('motors', {}).items():
        row[M] = f'{v["amplitude"]:.1f}" @ {v["phase"]:.0f}° ({v["significance"]:.0f}σ)'
    rows.append(row)
if rows:
    display(pd.DataFrame(rows).set_index('#').round(1))

use = pd.DataFrame([{'motor': M, **{f'h{h}': f'{np.hypot(*in_use(m)[2*h-2:2*h]):.1f}" @ '
                                         f'{np.degrees(np.arctan2(in_use(m)[2*h-1], in_use(m)[2*h-2])) % 360:.0f}°' for h in (1, 2)}}
                    for m, M in enumerate(MOTORS)]).set_index('motor')
applied = raw.get('applied_profile')
print(f"Profile in use: harmonics {list(ff.harmonics)}, angles {ff.angle_reference}" +
      (f", applied {applied['created']} from {applied['tests']} pooled tests" if applied else ' (not from a pooled test)'))
display(use)

Profile in use: harmonics [1], angles {'M1': 'zeta', 'M2': 'zeta', 'M3': 'zeta'} (not from a pooled test)


,h1,h2
motor,,
M1,"37.8"" @ 225°","0.0"" @ 0°"
M2,"64.0"" @ 281°","0.0"" @ 0°"
M3,"98.9"" @ 209°","0.0"" @ 0°"


## 2. Did a test measure the worms?

One test (`TEST`), sample by sample on both tangent axes: the measured error, the fit, and what each motor's worm
contributed. Then each motor's worm on its own: the fitted curve against the data with everything else (the other
motors, offset, drift, trend, backlash) removed, divided by that motor's effect on the axis it moves most. Points
scattering evenly around the curve, over the whole cycle, in both directions of travel, are a good measurement.

In [3]:
if tests:
    e = tests[TEST]
    d = test_fit_details(e['samples'])
    print(f"test {TEST % len(tests)} ({e['created']}): {e['status']}, {d['position'].nunique()} positions, "
          f"residual rms {d['residual'].std():.2f}\"")
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                        subplot_titles=['horizontal (arcsec)', 'up (arcsec)'])
    for a in range(2):
        x = d[d['axis'] == a]
        sl = a == 0
        fig.add_trace(go.Scatter(x=x['position'], y=x['measured'], mode='markers', name='measured', legendgroup='m',
                                 showlegend=sl, marker=dict(color='white', size=7)), row=a + 1, col=1)
        fig.add_trace(go.Scatter(x=x['position'], y=x['fitted'], name='fit', legendgroup='f', showlegend=sl,
                                 line=dict(color='red')), row=a + 1, col=1)
        for M in MOTORS:
            fig.add_trace(go.Scatter(x=x['position'], y=x[f'worm_{M}'], name=f'{M} worm', legendgroup=M, showlegend=sl,
                                     line=dict(color=COLOURS[M], dash='dot')), row=a + 1, col=1)
        fig.add_trace(go.Scatter(x=x['position'], y=x['nuisance'], name='offset, drift, trend, backlash', legendgroup='n',
                                 showlegend=sl, line=dict(color='gray', dash='dash')), row=a + 1, col=1)
    fig.update_xaxes(title_text='position', row=2, col=1)
    fig.update_layout(height=650, width=1300, template=TEMPLATE, hovermode='x unified', title='The test, sample by sample')
    fig.show()

    fig = make_subplots(rows=1, cols=3, subplot_titles=[f'{M}: worm alone (arcsec of motor angle)' for M in MOTORS])
    ph = np.linspace(0, 360, 181)
    for m, M in enumerate(MOTORS):
        # per sample, the axis this motor moves most: partial residual / J
        best = d.loc[d.groupby('position')[f'J_{M}'].apply(lambda s: s.abs().idxmax())]
        part = (best['residual'] + best[f'worm_{M}']) / best[f'J_{M}']
        phase = (360 * best[f'zeta_{M}'] / WORM_THETA) % 360
        for dirn, sym in ((1, 'triangle-up'), (-1, 'triangle-down')):
            k = best[f'dir_{M}'] == dirn
            fig.add_trace(go.Scatter(x=phase[k], y=part[k], mode='markers', marker=dict(symbol=sym, size=9, color=COLOURS[M]),
                                     name=f'{M} {"forward" if dirn > 0 else "back"}', showlegend=False), row=1, col=m + 1)
        v = e['result']['motors'][M]
        curve = v['amplitude'] * np.sin(np.radians(ph + v['phase'])) + \
            v.get('h2_amplitude', 0) * np.sin(np.radians(2 * ph + v.get('h2_phase', 0)))
        fig.add_trace(go.Scatter(x=ph, y=curve, line=dict(color='white'), showlegend=False), row=1, col=m + 1)
        fig.update_xaxes(title_text='worm phase (deg)', row=1, col=m + 1)
    fig.update_layout(height=420, width=1400, template=TEMPLATE,
                      title='Each motor\'s worm alone: ▲ moving forward, ▼ moving back, white = the fit (h1 + h2)')
    fig.show()

## 3. Do tests agree?

Each completed test's 1st harmonic per motor as a phasor (a₁, b₁), with a circle of 2 SE, and the pooled fit (what
Approve applies). Tests agree when their points sit within each other's circles. The table compares the spread of the
tests around the pooled fit with what their SE predicts: a ratio well above 1 means the SE is optimistic (on the twin
it is 1.5–2) and the spread is the honest uncertainty of a single test; the pooled profile's error is roughly that
spread / √tests.

In [4]:
done = [e for e in tests if e['status'] == 'COMPLETED']
pooled = pooled_profile(PROFILE) if done else {'status': 'NO DATA'}
if not done:
    print("No completed tests.")
else:
    print(f"pooled over the last {min(len(done), POOL_TESTS)} completed tests: {pooled['status']}, rms {pooled['checks']['rms_arcsec']}\"")
    fig = make_subplots(rows=1, cols=3, subplot_titles=[f'{M} 1st harmonic (arcsec)' for M in MOTORS])
    agree = []
    t_circle = np.linspace(0, 2 * np.pi, 60)
    for m, M in enumerate(MOTORS):
        pts = np.array([phasor(e['result']['motors'][M]['amplitude'], e['result']['motors'][M]['phase']) for e in done])
        ses = np.array([e['result']['motors'][M]['se'] for e in done])
        for i, ((a, b), se) in enumerate(zip(pts, ses)):
            fig.add_trace(go.Scatter(x=[a], y=[b], mode='markers+text', text=[str(tests.index(done[i]))], textposition='top center',
                                     marker=dict(color=COLOURS[M], size=9), showlegend=False), row=1, col=m + 1)
            fig.add_trace(go.Scatter(x=a + 2 * se * np.cos(t_circle), y=b + 2 * se * np.sin(t_circle), mode='lines',
                                     line=dict(color=COLOURS[M], width=0.6), showlegend=False), row=1, col=m + 1)
        pv = pooled['motors'][M]
        pa, pb = phasor(pv['amplitude'], pv['phase'])
        fig.add_trace(go.Scatter(x=[pa], y=[pb], mode='markers', marker=dict(symbol='star', size=16, color='white'),
                                 name='pooled', showlegend=m == 0), row=1, col=m + 1)
        cu = in_use(m)
        fig.add_trace(go.Scatter(x=[cu[0]], y=[cu[1]], mode='markers', marker=dict(symbol='x', size=13, color='red'),
                                 name='in use', showlegend=m == 0), row=1, col=m + 1)
        if TRUTH:
            fig.add_trace(go.Scatter(x=[TRUTH[M]['coef'][0]], y=[TRUTH[M]['coef'][1]], mode='markers',
                                     marker=dict(symbol='circle-open', size=18, color='yellow', line=dict(width=2)),
                                     name='twin truth', showlegend=m == 0), row=1, col=m + 1)
        dist = np.hypot(*(pts - [pa, pb]).T)
        spread = float(np.sqrt(np.mean(dist ** 2) * len(pts) / max(len(pts) - 1, 1)))
        agree.append({'motor': M, 'tests': len(pts), 'spread rms "': spread, 'median SE "': float(np.median(ses)),
                      'spread / (√2·SE)': spread / (np.sqrt(2) * np.median(ses)) if len(pts) > 2 else np.nan,
                      'pooled "': pv['amplitude'], 'pooled phase°': pv['phase'], 'pooled SE "': pv['se']})
        fig.add_hline(y=0, line=dict(color='gray', width=1), row=1, col=m + 1)
        fig.add_vline(x=0, line=dict(color='gray', width=1), row=1, col=m + 1)
        fig.update_xaxes(title_text='a₁ (sin)', row=1, col=m + 1)
        fig.update_yaxes(scaleanchor=f'x{m + 1 if m else ""}', row=1, col=m + 1)
    fig.update_layout(height=480, width=1450, template=TEMPLATE, title='Each test\'s 1st harmonic (circles: 2 SE)')
    fig.show()
    display(pd.DataFrame(agree).set_index('motor').round(2))

No completed tests.


## 4. Does the pose matter?

M1 and M3 move the field in directions that differ by about the roll angle; below ~10° their worms trade off against
each other and their errors grow together. Each completed test's distance from the pooled fit against its M1/M3
separation: with enough roll it shouldn't depend on the pose.

In [5]:
if done:
    pr = []
    for e in done:
        sep = e['result']['checks'].get('separation_deg', [np.nan, np.nan])
        for m, M in enumerate(MOTORS):
            a, b = phasor(e['result']['motors'][M]['amplitude'], e['result']['motors'][M]['phase'])
            pa, pb = phasor(pooled['motors'][M]['amplitude'], pooled['motors'][M]['phase'])
            pr.append({'test': tests.index(e), 'motor': M, 'separation min°': sep[0], 'separation max°': sep[1],
                       'alt': np.mean([x['a_alt'] for x in e['samples']]), 'az': np.mean([x['a_az'] for x in e['samples']]),
                       'from pooled "': np.hypot(a - pa, b - pb), 'SE "': e['result']['motors'][M]['se']})
    pr = pd.DataFrame(pr)
    fig = go.Figure()
    for M in MOTORS:
        x = pr[pr['motor'] == M]
        fig.add_trace(go.Scatter(x=x['separation min°'], y=x['from pooled "'], mode='markers+text', text=x['test'],
                                 textposition='top center', name=M, marker=dict(color=COLOURS[M], size=10),
                                 error_y=dict(array=x['SE "'])))
    fig.add_vline(x=MIN_SEPARATION_DEG, line=dict(color='red', dash='dot'))
    fig.update_layout(height=420, width=900, template=TEMPLATE, xaxis_title='M1/M3 separation, least during the test (deg)',
                      yaxis_title='distance from the pooled fit (arcsec)', title='Test error against the pose')
    fig.show()
    display(pr.pivot_table(index='test', values=['separation min°', 'alt', 'az'], aggfunc='first').round(1))

## 5. Is the model right?

* **2nd harmonic:** its amplitude and significance per test; applied only when ≥ 3 SE. If a motor's 2nd harmonic is
  consistently significant, it is real (a 2nd harmonic on M2 of ~20″ has been seen on the real mount).
* **Backlash:** the offset each motor shows moving forward against moving back, in arcsec of motor angle. It is
  fitted so it doesn't bias the worm, and reported only: tracking turns each motor one way, so the feed-forward
  doesn't apply it.
* **Fit rms with and without the 2nd harmonic:** how much of the error it explains.
* **Residuals by direction:** whatever the model misses, per motor direction of travel.

In [6]:
if tests:
    rows = []
    for i, e in enumerate(tests):
        r1 = fit_worm_profile([e['samples']], harmonics=(1,))
        row = {'#': i, 'status': e['status'], 'rms h1+h2 "': e['result']['checks'].get('rms_arcsec'),
               'rms h1 only "': r1['checks'].get('rms_arcsec')}
        for M, v in e['result'].get('motors', {}).items():
            row[f'{M} h2'] = f'{v["h2_amplitude"]:.1f}" ({v["h2_amplitude"] / max(v["h2_se"], 1e-9):.1f}σ)'
            row[f'{M} backlash "'] = v['backlash']
        rows.append(row)
    display(pd.DataFrame(rows).set_index('#'))
    if done:
        print('pooled:', {M: f'h2 {v["h2_amplitude"]:.1f}" ± {v["h2_se"]:.1f} @ {v["h2_phase"]:.0f}°, backlash {v["backlash"]:+.1f}"'
                          for M, v in pooled['motors'].items()})

    d = test_fit_details(tests[TEST]['samples'])
    fig = make_subplots(rows=1, cols=3, subplot_titles=[f'{M}: residual by direction of travel' for M in MOTORS])
    for m, M in enumerate(MOTORS):
        for dirn, c in ((1, 'deepskyblue'), (-1, 'white')):
            x = d[d[f'dir_{M}'] == dirn]
            fig.add_trace(go.Box(y=x['residual'], name='forward' if dirn > 0 else 'back', marker_color=c,
                                 showlegend=False), row=1, col=m + 1)
    fig.update_yaxes(title_text='arcsec', row=1, col=1)
    fig.update_layout(height=380, width=1200, template=TEMPLATE, title=f'Test {TEST % len(tests)}: what the fit leaves')
    fig.show()

## 6. What would Approve change?

Approve applies the fit pooled over the last 5 completed tests (both harmonics, the 2nd only where significant);
Reject restores the profile before it. *Change* is the worm the current profile leaves that the pooled one would
correct, per motor (the distance between the two phasors, 1st and 2nd harmonic together).

In [7]:
if done and pooled['status'] == 'COMPLETED':
    rows = []
    for m, M in enumerate(MOTORS):
        new = np.zeros(4)
        c = pooled['motors'][M]['coef']
        new[:len(c)] = c
        old = in_use(m)
        rows.append({'motor': M, 'in use h1': f'{np.hypot(*old[:2]):.1f}" @ {np.degrees(np.arctan2(old[1], old[0])) % 360:.0f}°',
                     'pooled h1': f'{np.hypot(*new[:2]):.1f}" @ {np.degrees(np.arctan2(new[1], new[0])) % 360:.0f}°',
                     'in use h2 "': np.hypot(*old[2:]), 'pooled h2 "': np.hypot(*new[2:]),
                     'change "': float(np.sqrt(np.sum((new - old) ** 2) / 2))})
    display(pd.DataFrame(rows).set_index('motor').round(1))
    if applied:
        print(f"Already applied {applied['created']}; Reject would restore: "
              + ', '.join(f'{M} {np.hypot(*applied["previous"]["motors"][M][:2]):.1f}"' for M in MOTORS))
else:
    print("Nothing to approve: no completed tests.")

Nothing to approve: no completed tests.


## 7. How long does a test take?

A sync is kept once every motor has held within 10″ of its target for 1 s after the step; earlier syncs are
discarded (*moving*), and so is one whose error jumps by over 300″ (its exposure caught the move). With the client's
wait between solves at least the settle time, every solve is kept: 33 solves per test.

In [8]:
if tests:
    rows = []
    for i, e in enumerate(tests):
        s, tm = e['samples'], e.get('timing') or {}
        t = np.array([x['t'] for x in s])
        st = [x['settle_s'] for x in s if x.get('settle_s') is not None]
        since = [x['since_settle_s'] for x in s if x.get('since_settle_s') is not None]
        rows.append({'#': i, 'kept': len(s), 'minutes': np.ptp(t) / 60, 's per position': np.median(np.diff(t)) if len(t) > 1 else np.nan,
                     'settle median s': tm.get('settle_median_s'), 'settle max s': tm.get('settle_max_s'),
                     'sync after settled, median s': np.median(since) if since else np.nan,
                     'discarded moving': (tm.get('discarded') or {}).get('moving'),
                     'discarded jump': (tm.get('discarded') or {}).get('jump')})
    display(pd.DataFrame(rows).set_index('#').round(1))

## 8. Does a night of tracking agree?

During tracking the feed-forward logs its correction (PECLOG `wff`, RA/Dec arcmin) from each motor's angle and the
pose. Fitting each motor's worm in a session's corrections (as in `analyse_tracking.ipynb` section 7):

* **the logged wff** returns the profile that was in use (a check that the log and the profile agree);
* **guide corrections + wff** is what the mount needed: an independent estimate of the worms, from tracking.

Tracking turns M1 fast (about 12 worm periods an hour near the pole) but M2 and M3 slowly, so their tracking
estimates are weak unless the night was long: check *turns*. The amplitudes' SE here is optimistic too. The phasor
plot puts the tracking estimate next to the calibration tests.

How strongly each motor's worm reaches RA and Dec depends on the pose; the coupling plot shows it over the night.

In [9]:
TRACKING_SESSION = None if TRUTH else 'jdm_Beta7.3_10_04_worm'    # a twin file's worms aren't the real mount's
TRACKING_FROM, TRACKING_TO = '2026-10-04 21:34', None                # a window with one profile in use

if TRACKING_SESSION:
    from sessions import get_session, site_latitude
    from analyse_helpers import load_pec, window_df
    from extract_segments import DEFAULT_SITE
    S = get_session(TRACKING_SESSION)
    pec, _ = load_pec(S.log_filenames, log_dir=S.log_dir)
    w = window_df(pec, TRACKING_FROM, TRACKING_TO).dropna(subset=['theta_raw_1', 'wff_1', 'total_accum_1', 'total_accum_2'])
    w = w.reset_index(drop=True)
    C, align_err = wff_coupling(w, site_latitude(TRACKING_SESSION, default=DEFAULT_SITE[0]))
    turns = [round(float(x), 1) for x in np.ptp(motor_angles(w), axis=0) / WORM_THETA]
    wff = w[['wff_1', 'wff_2']].to_numpy(float)
    guide = np.column_stack([w[f'total_accum_{i}'].to_numpy(float) + w[f'pec_accum_{i}'].fillna(0).to_numpy(float) for i in (1, 2)])
    est = {}
    for label, y in (('logged wff', wff), ('needed (guide + wff)', guide + wff)):
        terms, rms_ = worm_terms(w, y, C)
        est[label] = terms.set_index('motor')
    print(f"{TRACKING_SESSION} {w['timestamp'].iloc[0]:%H:%M}-{w['timestamp'].iloc[-1]:%H:%M}, {len(w)} rows, worm turns {turns}")
    display(pd.concat({k: v[['amplitude', 'phase', 'se']] for k, v in est.items()}, axis=1).round(1))

    fig = make_subplots(rows=1, cols=3, subplot_titles=[f'{M} 1st harmonic (arcsec)' for M in MOTORS])
    for m, M in enumerate(MOTORS):
        for e in done:
            a, b = phasor(e['result']['motors'][M]['amplitude'], e['result']['motors'][M]['phase'])
            fig.add_trace(go.Scatter(x=[a], y=[b], mode='markers', marker=dict(color=COLOURS[M], size=8),
                                     name='calibration tests', showlegend=m == 0 and e is done[0]), row=1, col=m + 1)
        for label, sym, c in (('logged wff', 'x', 'red'), ('needed (guide + wff)', 'diamond', 'white')):
            r = est[label].loc[M]
            fig.add_trace(go.Scatter(x=[r['a']], y=[r['b']], mode='markers', marker=dict(symbol=sym, size=13, color=c),
                                     error_x=dict(array=[r['se']]), error_y=dict(array=[r['se']]),
                                     name=f'tracking: {label}', showlegend=m == 0), row=1, col=m + 1)
        fig.add_hline(y=0, line=dict(color='gray', width=1), row=1, col=m + 1)
        fig.add_vline(x=0, line=dict(color='gray', width=1), row=1, col=m + 1)
        fig.update_yaxes(scaleanchor=f'x{m + 1 if m else ""}', row=1, col=m + 1)
    fig.update_layout(height=480, width=1450, template=TEMPLATE, title='Tracking estimates against the calibration tests')
    fig.show()

    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.1,
                        subplot_titles=['RA arcmin per deg of motor angle', 'Dec arcmin per deg of motor angle'])
    for k in range(2):
        for m, M in enumerate(MOTORS):
            fig.add_trace(go.Scatter(x=w['timestamp'], y=C[:, m, k], name=M, legendgroup=M, showlegend=k == 0,
                                     line=dict(color=COLOURS[M])), row=k + 1, col=1)
    fig.update_layout(height=550, width=1300, template=TEMPLATE, hovermode='x unified',
                      title='How much each motor\'s worm moves RA and Dec over the night')
    fig.show()
else:
    print("No tracking session to compare (TRACKING_SESSION = None).")

jdm_Beta7.3_10_04_worm 21:34-22:33, 2713 rows, worm turns [12.5, 1.6, 1.8]


logged wff             needed (guide + wff)            
       amplitude  phase   se            amplitude  phase   se
motor                                                        
M1          37.7  224.9  0.0                 34.0  254.1  0.7
M2          64.4  280.5  0.0                 42.9  311.7  0.5
M3          98.8  208.6  0.0                127.3  201.6  0.6

## 9. Twin only: do the fits recover the simulated worms?

With a file written by the digital twin (`tests/sim_digital_twin.py`, worms the driver can't see), each test's error
against the true worm, and that error in units of the fit's SE. Honest error bars would give errors / SE around 1
(2-D: around √2 for the distance); twin runs have shown up to 1.5–2 × that, the pooled fit's most.

In [10]:
if not TRUTH:
    print("Not a twin file: no true worms to compare.")
else:
    rows = []
    for i, e in enumerate(tests + ([{'created': 'pooled', 'result': pooled, 'status': pooled['status']}] if done else [])):
        for M in MOTORS:
            v = e['result']['motors'][M]
            a, b = phasor(v['amplitude'], v['phase'])
            ta, tb = TRUTH[M]['coef'][:2]
            err = float(np.hypot(a - ta, b - tb))
            rows.append({'test': e['created'], 'motor': M, 'h1 error "': err, 'SE "': v['se'], 'error / SE': err / v['se'],
                         'h2 error "': float(np.hypot(*(np.asarray(v['coef'][2:4] if len(v['coef']) > 2 else [0, 0]) - TRUTH[M]['coef'][2:4])))})
    r = pd.DataFrame(rows)
    display(r.pivot(index='test', columns='motor', values=['h1 error "', 'error / SE']).round(2))
    single = r[r['test'] != 'pooled']
    print(f"single tests: h1 error rms {np.sqrt(np.mean(single['h1 error \"'] ** 2)):.1f}\", "
          f"error/SE rms {np.sqrt(np.mean(single['error / SE'] ** 2)):.2f} (honest: ~1.4)")

Not a twin file: no true worms to compare.
